In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def make_sample(label, size=16):
    img = np.zeros((size, size), dtype=float)
    shift = rng.integers(-2, 3)
    if label == 0:
        r = np.clip(size//2 + shift, 2, size-3)
        img[r-1:r+2, :] = 1.0
    else:
        c = np.clip(size//2 + shift, 2, size-3)
        img[:, c-1:c+2] = 1.0
    img += rng.normal(0, 0.1, img.shape)
    return np.clip(img, 0, 1)

def make_dataset(n=1000):
    X = np.zeros((n,16,16), dtype=float)
    y = np.zeros(n, dtype=int)
    for i in range(n):
        lbl = rng.integers(0,2)
        X[i] = make_sample(lbl)
        y[i] = lbl
    return X, y

X, y = make_dataset(1200)
perm = rng.permutation(len(X))
X, y = X[perm], y[perm]
split = int(0.8*len(X))
Xtr, Xva = X[:split], X[split:]
ytr, yva = y[:split], y[split:]

def one_hot(y, C):
    Y = np.zeros((y.size, C), dtype=float)
    Y[np.arange(y.size), y] = 1.0
    return Y

C = 2
Ytr, Yva = one_hot(ytr, C), one_hot(yva, C)

f = 3
Wc = rng.normal(0, 0.1, (f,f))
bc = np.zeros((1,))
flat_dim = 7*7
Wd = rng.normal(0, 0.1, (flat_dim, C))
bd = np.zeros((1,C))

def conv2d_valid(X, W, b):
    B,H,Wid = X.shape
    f = W.shape[0]
    Ho, Wo = H-f+1, Wid-f+1
    out = np.zeros((B, Ho, Wo), dtype=float)
    for i in range(Ho):
        for j in range(Wo):
            region = X[:, i:i+f, j:j+f]
            out[:, i, j] = np.tensordot(region, W, axes=((1,2),(0,1))) + b
    return out

def relu(x): return np.maximum(0, x)
def relu_deriv(x): return (x > 0).astype(x.dtype)

def maxpool2x2(X):
    B,H,Wid = X.shape
    Ho, Wo = H//2, Wid//2
    out = np.zeros((B, Ho, Wo), dtype=float)
    mask = np.zeros_like(X, dtype=bool)
    for i in range(Ho):
        for j in range(Wo):
            patch = X[:, 2*i:2*i+2, 2*j:2*j+2]
            m = patch.reshape(B, -1).argmax(axis=1)
            out[:, i, j] = patch.reshape(B, -1)[np.arange(B), m]
            r, c = m//2, m%2
            mask[np.arange(B), 2*i+r, 2*j+c] = True
    return out, mask

def maxpool2x2_back(grad_out, mask):
    B,H,W = mask.shape
    Ho, Wo = grad_out.shape[1:3]
    grad = np.zeros((B,H,W), dtype=float)
    for i in range(Ho):
        for j in range(Wo):
            g = grad_out[:, i, j]
            block = mask[:, 2*i:2*i+2, 2*j:2*j+2]
            idx = block.reshape(B, -1).argmax(axis=1)
            r, c = idx//2, idx%2
            grad[np.arange(B), 2*i+r, 2*j+c] = g
    return grad

def softmax(z):
    z_shift = z - z.max(axis=1, keepdims=True)
    expz = np.exp(z_shift)
    return expz / (expz.sum(axis=1, keepdims=True)+1e-12)

def cross_entropy(y_true, y_prob):
    return -np.mean((y_true * np.log(y_prob+1e-12)).sum(axis=1))

def accuracy(y_true_labels, y_prob):
    return np.mean(y_true_labels == y_prob.argmax(axis=1))

lr = 0.05
epochs = 40
batch = 64

for ep in range(1, epochs+1):
    idx = rng.permutation(len(Xtr))
    Xtr_b, Ytr_b = Xtr[idx], Ytr[idx]
    for s in range(0, len(Xtr_b), batch):
        Xe = Xtr_b[s:s+batch]
        Ye = Ytr_b[s:s+batch]
        B = Xe.shape[0]
        zc = conv2d_valid(Xe, Wc, bc)
        ac = relu(zc)
        pm, mask = maxpool2x2(ac)
        flat = pm.reshape(B, -1)
        logits = flat @ Wd + bd
        prob = softmax(logits)
        dlogits = (prob - Ye)/B
        dWd = flat.T @ dlogits
        dbd = dlogits.sum(axis=0, keepdims=True)
        dflat = dlogits @ Wd.T
        dpm = dflat.reshape(B,7,7)
        dac = maxpool2x2_back(dpm, mask)
        dzc = dac * relu_deriv(zc)
        dWc = np.zeros_like(Wc)
        dbc = dzc.sum(axis=(0,1,2))
        for i in range(14):
            for j in range(14):
                region = Xe[:, i:i+f, j:j+f]
                dWc += (region * dzc[:, i:i+1, j:j+1]).sum(axis=0)
        Wd -= lr*dWd
        bd -= lr*dbd
        Wc -= lr*dWc
        bc -= lr*dbc
    zc = conv2d_valid(Xtr, Wc, bc)
    ac = relu(zc)
    pm, _ = maxpool2x2(ac)
    flat = pm.reshape(len(Xtr), -1)
    prob_tr = softmax(flat @ Wd + bd)
    loss_tr = cross_entropy(Ytr, prob_tr)
    acc_tr = accuracy(ytr, prob_tr)
    zc = conv2d_valid(Xva, Wc, bc)
    ac = relu(zc)
    pm, _ = maxpool2x2(ac)
    flat = pm.reshape(len(Xva), -1)
    prob_va = softmax(flat @ Wd + bd)
    loss_va = cross_entropy(Yva, prob_va)
    acc_va = accuracy(yva, prob_va)
    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:2d} | loss_tr {loss_tr:.3f} acc_tr {acc_tr:.3f} | loss_va {loss_va:.3f} acc_va {acc_va:.3f}")

print("\nSample validation predictions:")
for i in range(5):
    p = prob_va[i].argmax()
    print(f"True={yva[i]} Pred={p} Prob={prob_va[i].round(3).tolist()}")
